# IGTS Portfolio Game — Week 1 Solutions
**Indian Game Theory Society, IIT Kanpur**

In [ ]:
# Question 1 — Reading Payoff Matrices
#
# Payoff Matrix:
#           Left      Right
#  Top    (3, 3)    (0, 5)
#  Bottom (5, 0)    (1, 1)
#
# (a) P1 plays Top, P2 plays Right → cell (Top, Right) = (0, 5)
#     Player 1 receives: 0
#
# (b) Both players play Bottom → cell (Bottom, Right) = (1, 1)
#     Player 2 receives: 1
#
# (c) When P2 plays Left  → P1 gets 3 (Top) vs 5 (Bottom) → Bottom gives higher payoff
#     When P2 plays Right → P1 gets 0 (Top) vs 1 (Bottom) → Bottom gives higher payoff
#     Note: Bottom dominates Top for Player 1 regardless of P2's choice.
#     This is also the classic Prisoner's Dilemma structure!

In [ ]:
# Question 2 — Dominant Strategy Finder (2-Player)

def find_dominant_strategy(matrix):
    """
    Finds strictly dominant strategies for both players in a 2-player game.
    matrix: 2D list of (p1_payoff, p2_payoff) tuples.
    """
    num_rows = len(matrix)
    num_cols = len(matrix[0])

    print("=" * 50)
    print("Dominant Strategy Analysis")
    print("=" * 50)

    # --- Player 1: check rows ---
    # A row r* strictly dominates row r if p1_payoff[r*][c] > p1_payoff[r][c] for ALL columns c
    p1_dominant = None
    for r_star in range(num_rows):
        dominates_all = True
        for r in range(num_rows):
            if r == r_star:
                continue
            # r_star must beat r in every column
            if not all(matrix[r_star][c][0] > matrix[r][c][0] for c in range(num_cols)):
                dominates_all = False
                break
        if dominates_all:
            p1_dominant = r_star
            break

    if p1_dominant is not None:
        print(f"Player 1 has a strictly dominant strategy: Row {p1_dominant}")
    else:
        print("Player 1 has NO strictly dominant strategy.")

    # --- Player 2: check columns ---
    # A column c* strictly dominates column c if p2_payoff[r][c*] > p2_payoff[r][c] for ALL rows r
    p2_dominant = None
    for c_star in range(num_cols):
        dominates_all = True
        for c in range(num_cols):
            if c == c_star:
                continue
            if not all(matrix[r][c_star][1] > matrix[r][c][1] for r in range(num_rows)):
                dominates_all = False
                break
        if dominates_all:
            p2_dominant = c_star
            break

    if p2_dominant is not None:
        print(f"Player 2 has a strictly dominant strategy: Column {p2_dominant}")
    else:
        print("Player 2 has NO strictly dominant strategy.")

    print()


# --- Test Matrices ---

# (i) Advertise / Not-Advertise
#       Advertise   Not-Advertise
# Adv   (4, 4)      (6, 1)
# Not   (1, 6)      (3, 3)
print("(i) Advertise / Not-Advertise")
advertise_matrix = [
    [(4, 4), (6, 1)],
    [(1, 6), (3, 3)]
]
find_dominant_strategy(advertise_matrix)

# (ii) Prisoner's Dilemma
#        Cooperate   Defect
# Coop   (-1, -1)   (-3,  0)
# Def    ( 0, -3)   (-2, -2)
print("(ii) Prisoner's Dilemma")
prisoners_dilemma = [
    [(-1, -1), (-3,  0)],
    [( 0, -3), (-2, -2)]
]
find_dominant_strategy(prisoners_dilemma)

# (iii) Stag Hunt
#        Stag   Hare
# Stag   (4,4)  (0,3)
# Hare   (3,0)  (3,3)
print("(iii) Stag Hunt")
stag_hunt = [
    [(4, 4), (0, 3)],
    [(3, 0), (3, 3)]
]
find_dominant_strategy(stag_hunt)

In [ ]:
# Question 3 — IESDS Solver (2-Player)

def iesds(matrix, row_labels=None, col_labels=None):
    """
    Iterated Elimination of Strictly Dominated Strategies.
    matrix      : 2D list of (p1_payoff, p2_payoff) tuples
    row_labels  : optional list of names for P1's strategies
    col_labels  : optional list of names for P2's strategies
    """
    import copy

    # Work with index trackers so we can report original labels
    rows = list(range(len(matrix)))
    cols = list(range(len(matrix[0])))
    m    = copy.deepcopy(matrix)      # we'll never mutate; use index subsets

    if row_labels is None:
        row_labels = [f"Row {r}" for r in rows]
    if col_labels is None:
        col_labels = [f"Col {c}" for c in cols]

    print("=" * 55)
    print("IESDS Solver")
    print("=" * 55)

    iteration = 0
    changed   = True

    while changed:
        changed = False

        # --- Check P1's rows ---
        for r in list(rows):
            for r2 in rows:
                if r == r2:
                    continue
                # Does r2 strictly dominate r? (r2 beats r in every surviving column)
                if all(m[r2][c][0] > m[r][c][0] for c in cols):
                    iteration += 1
                    print(f"Iteration {iteration}: Eliminate P1 strategy '{row_labels[r]}' "
                          f"— strictly dominated by '{row_labels[r2]}'")
                    rows.remove(r)
                    changed = True
                    break
            if changed:
                break

        if changed:
            continue  # restart scan after each elimination

        # --- Check P2's columns ---
        for c in list(cols):
            for c2 in cols:
                if c == c2:
                    continue
                # Does c2 strictly dominate c?
                if all(m[r][c2][1] > m[r][c][1] for r in rows):
                    iteration += 1
                    print(f"Iteration {iteration}: Eliminate P2 strategy '{col_labels[c]}' "
                          f"— strictly dominated by '{col_labels[c2]}'")
                    cols.remove(c)
                    changed = True
                    break
            if changed:
                break

    print()
    print("Surviving strategies:")
    print(f"  Player 1: {[row_labels[r] for r in rows]}")
    print(f"  Player 2: {[col_labels[c] for c in cols]}")

    if len(rows) == 1 and len(cols) == 1:
        cell = m[rows[0]][cols[0]]
        print(f"\nPredicted outcome: ({row_labels[rows[0]]}, {col_labels[cols[0]]}) "
              f"with payoffs {cell}")
    print()


# --- Test: 3×4 matrix from slides (A/B/C vs D/E/F/G) ---
# Expected elimination order leads to (B, D)
#
#         D       E       F       G
# A    (3,3)   (0,4)   (1,3)   (0,3)
# B    (4,0)   (3,2)   (2,3)   (5,1)
# C    (2,5)   (1,3)   (3,2)   (0,4)

matrix_3x4 = [
    [(3,3), (0,4), (1,3), (0,3)],   # A
    [(4,0), (3,2), (2,3), (5,1)],   # B
    [(2,5), (1,3), (3,2), (0,4)],   # C
]

iesds(matrix_3x4,
      row_labels=['A', 'B', 'C'],
      col_labels=['D', 'E', 'F', 'G'])

In [ ]:
# Question 4 — Nash Equilibrium Finder: Pure Strategies (2-Player)

def find_nash_pure(matrix, row_labels=None, col_labels=None, game_name=""):
    """
    Finds all pure-strategy Nash Equilibria using the best-response method.
    """
    num_rows = len(matrix)
    num_cols = len(matrix[0])

    if row_labels is None:
        row_labels = [f"Row {r}" for r in range(num_rows)]
    if col_labels is None:
        col_labels = [f"Col {c}" for c in range(num_cols)]

    # --- Step (a): Best responses for Player 1 ---
    # For each fixed column c, find which row(s) maximise P1's payoff
    p1_best = set()
    for c in range(num_cols):
        max_p1 = max(matrix[r][c][0] for r in range(num_rows))
        for r in range(num_rows):
            if matrix[r][c][0] == max_p1:
                p1_best.add((r, c))

    # --- Step (b): Best responses for Player 2 ---
    # For each fixed row r, find which column(s) maximise P2's payoff
    p2_best = set()
    for r in range(num_rows):
        max_p2 = max(matrix[r][c][1] for c in range(num_cols))
        for c in range(num_cols):
            if matrix[r][c][1] == max_p2:
                p2_best.add((r, c))

    # --- Step (c): Nash Equilibria = intersection ---
    nash_cells = p1_best & p2_best

    header = f"Nash Equilibria — {game_name}" if game_name else "Nash Equilibria"
    print("=" * 55)
    print(header)
    print("=" * 55)

    if nash_cells:
        for (r, c) in sorted(nash_cells):
            print(f"  NE at ({row_labels[r]}, {col_labels[c]}) "
                  f"with payoffs {matrix[r][c]}")
    else:
        print("  No pure-strategy Nash Equilibrium found.")
    print()


# --- (i) Prisoner's Dilemma ---
prisoners_dilemma = [
    [(-1, -1), (-3,  0)],
    [( 0, -3), (-2, -2)]
]
find_nash_pure(prisoners_dilemma,
               row_labels=['Cooperate','Defect'],
               col_labels=['Cooperate','Defect'],
               game_name="Prisoner's Dilemma")

# --- (ii) Stag Hunt ---
stag_hunt = [
    [(4, 4), (0, 3)],
    [(3, 0), (3, 3)]
]
find_nash_pure(stag_hunt,
               row_labels=['Stag','Hare'],
               col_labels=['Stag','Hare'],
               game_name="Stag Hunt")

# --- (iii) Chicken ---
#           Swerve   Straight
# Swerve    (0,0)    (-1, 1)
# Straight  (1,-1)   (-10,-10)
chicken = [
    [(0,  0), (-1,  1)],
    [(1, -1), (-10,-10)]
]
find_nash_pure(chicken,
               row_labels=['Swerve','Straight'],
               col_labels=['Swerve','Straight'],
               game_name="Chicken")

# --- (iv) Battle of the Sexes ---
#          Opera   Football
# Opera    (2,1)   (0,0)
# Football (0,0)   (1,2)
bos = [
    [(2, 1), (0, 0)],
    [(0, 0), (1, 2)]
]
find_nash_pure(bos,
               row_labels=['Opera','Football'],
               col_labels=['Opera','Football'],
               game_name="Battle of the Sexes")

# --- (v) 3x3 matrix from slides (A/B/C vs X/Y/Z) ---
#        X       Y       Z
# A   (3,2)   (1,4)   (2,3)
# B   (2,5)   (4,2)   (1,3)
# C   (5,1)   (2,3)   (3,2)
matrix_3x3 = [
    [(3,2), (1,4), (2,3)],
    [(2,5), (4,2), (1,3)],
    [(5,1), (2,3), (3,2)]
]
find_nash_pure(matrix_3x3,
               row_labels=['A','B','C'],
               col_labels=['X','Y','Z'],
               game_name="3x3 Slides Matrix")

In [ ]:
# Question 5 — Nash Equilibrium Finder: N-Player Game

import itertools

def find_nash_n_player(strategies, payoffs):
    """
    Finds all pure-strategy Nash Equilibria for an N-player game.
    strategies : list of lists — strategies[i] = strategy names for player i
    payoffs    : dict mapping strategy profile tuple -> list of payoffs (one per player)
    """
    n = len(strategies)
    nash_equilibria = []

    # (a) Enumerate ALL strategy profiles
    all_profiles = list(itertools.product(*strategies))

    for profile in all_profiles:
        is_nash = True

        # (b) For each player i, check if any deviation improves their payoff
        for i in range(n):
            current_payoff = payoffs[profile][i]

            for alt_strategy in strategies[i]:
                if alt_strategy == profile[i]:
                    continue  # same strategy, skip
                # Build the deviated profile
                deviated = list(profile)
                deviated[i] = alt_strategy
                deviated = tuple(deviated)

                if payoffs[deviated][i] > current_payoff:
                    is_nash = False  # player i can profitably deviate
                    break

            if not is_nash:
                break

        if is_nash:
            nash_equilibria.append(profile)

    # (c) Print results
    print("=" * 55)
    print("N-Player Nash Equilibrium Finder")
    print("=" * 55)
    if nash_equilibria:
        for ne in nash_equilibria:
            print(f"  NE: {ne}  |  Payoffs: {payoffs[ne]}")
    else:
        print("  No pure-strategy Nash Equilibrium exists.")
    print()


# --- Test: 3-Firm Collective Action Game ---
strategies = [['C','D'], ['C','D'], ['C','D']]
payoffs = {
    ('C','C','C'): [4, 4, 4],
    ('C','C','D'): [1, 1, 6],
    ('C','D','C'): [1, 6, 1],
    ('C','D','D'): [0, 3, 3],
    ('D','C','C'): [6, 1, 1],
    ('D','C','D'): [3, 0, 3],
    ('D','D','C'): [3, 3, 0],
    ('D','D','D'): [2, 2, 2]
}

find_nash_n_player(strategies, payoffs)

# --- Discussion (2–3 sentences) ---
# The only Nash Equilibrium is (D, D, D) with payoffs [2, 2, 2].
# This is not surprising — it mirrors the classic N-player Prisoner's Dilemma:
# each firm has an individual incentive to defect regardless of what the others do,
# even though universal cooperation (C,C,C) yields [4,4,4] for everyone.
# The result reveals the tragedy of the commons: rational self-interest
# leads to a collectively inferior outcome, highlighting why
# binding agreements or external enforcement are often needed in collective action problems.
print("Discussion:")
print("The sole NE is (D,D,D) with payoffs [2,2,2], even though (C,C,C) gives [4,4,4].")
print("Each firm can always gain by unilaterally defecting, so mutual defection is the")
print("only self-enforcing outcome — a classic N-player Prisoner's Dilemma / tragedy of the commons.")
print("This implies collective action problems require external enforcement or binding contracts to achieve cooperation.")

In [ ]:
# Question 6 — Interactive Game Solver

def interactive_game_solver():
    """
    Allows the user to input a custom payoff matrix at runtime,
    then runs dominant strategy detection, IESDS, and Nash Equilibrium finding.
    """

    # --- Reuse functions from previous questions ---
    # (Paste or import find_dominant_strategy, iesds, find_nash_pure here if running standalone)

    print("=" * 55)
    print("Welcome to the Interactive Two-Player Game Solver")
    print("=" * 55)

    # (a) Get matrix dimensions
    m = int(input("Enter number of strategies for Player 1 (rows): "))
    n = int(input("Enter number of strategies for Player 2 (columns): "))

    # (b) Input payoffs row by row
    matrix = []
    print("\nEnter payoffs row by row.")
    print("For each cell enter two space-separated integers: p1_payoff p2_payoff")
    for i in range(m):
        row = []
        for j in range(n):
            while True:
                raw = input(f"  Cell (Row {i}, Col {j}): ")
                parts = raw.strip().split()
                if len(parts) == 2:
                    try:
                        row.append((int(parts[0]), int(parts[1])))
                        break
                    except ValueError:
                        pass
                print("  Please enter exactly two integers separated by a space.")
        matrix.append(row)

    # (c) Display matrix
    print("\nYour Payoff Matrix:")
    header = "       " + "".join(f"  Col{j:<6}" for j in range(n))
    print(header)
    for i, row in enumerate(matrix):
        row_str = f"Row {i}  " + "".join(f"{str(cell):<10}" for cell in row)
        print(row_str)
    print()

    # (d) Run all three analyses
    print("\n--- Analysis 1: Dominant Strategies ---")
    find_dominant_strategy(matrix)

    print("--- Analysis 2: IESDS ---")
    iesds(matrix)

    print("--- Analysis 3: Pure-Strategy Nash Equilibria ---")
    find_nash_pure(matrix)


# --- Test: Battle of the Sexes (run interactively) ---
# When prompted, enter:
#   rows = 2, cols = 2
#   Cell (0,0): 2 1
#   Cell (0,1): 0 0
#   Cell (1,0): 0 0
#   Cell (1,1): 1 2
#
# Expected output:
#   No dominant strategy for either player.
#   IESDS: no eliminations possible (both NE survive).
#   Nash Equilibria: (Row 0, Col 0) = (2,1) and (Row 1, Col 1) = (1,2)

interactive_game_solver()